In [5]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [_here, *_here.parents]:
    if (candidate / "run_molecule.py").exists() and (candidate / "fm_graphex").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate Flowette project root. "
        "Run this notebook from inside the Flowette project."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

In [2]:
from run_molecule import main as run_molecule

In [3]:
TASK = "zinc" # Choose from zinc, qm9, moses, guacamol

# Set SMOKE_RUN=True only to verify the installation quickly.
SMOKE_RUN = False
FORCE_RETRAIN = True

if SMOKE_RUN:
    MAX_GRAPHS = 5000
    AE_EPOCHS = 10
    FM_EPOCHS = 50
    NUM_SAMPLES = 200
    SKIP_EVAL = True
else:
    # Set MAX_GRAPHS to an integer to use a subset of the original training dataset.
    # Set MAX_GRAPHS=None to use the entire training dataset.
    MAX_GRAPHS = None
    AE_EPOCHS = 200
    FM_EPOCHS = 500
    NUM_SAMPLES = 1000
    SKIP_EVAL = False

print({
    "task": TASK,
    "max_graphs": MAX_GRAPHS,
    "ae_epochs": AE_EPOCHS,
    "fm_epochs": FM_EPOCHS,
    "samples": NUM_SAMPLES,
})

{'task': 'zinc', 'max_graphs': 5000, 'ae_epochs': 10, 'fm_epochs': 10, 'samples': 200}


In [6]:
argv = [
    "--task", TASK,
    "--max-num-graphs", str(MAX_GRAPHS),
    "--ae-epochs", str(AE_EPOCHS),
    "--fm-epochs", str(FM_EPOCHS),
    "--fm-layers", "5",
    "--pairing-strategy", "online", # auto
    "--pairing-refresh-every", "5",
    "--n-steps", "200",
    "--num-samples", str(NUM_SAMPLES),
]
if FORCE_RETRAIN:
    argv.append("--force-train-fm")
if SKIP_EVAL:
    argv.append("--skip-eval")

generated_molecules, metrics = run_molecule(argv)

In [ ]:
from rdkit import Chem

valid_smiles = [
    Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    for m in generated_molecules
    if m is not None
]
print(f"Valid decoded molecules: {len(valid_smiles)}/{len(generated_molecules)}")
print("First 200 SMILES:")
for smi in valid_smiles[:200]:
    print(smi)
print("Metrics:", metrics)

In [ ]:
import os
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

IPythonConsole.drawOptions.maxMols = 1000

valid_mols = [m for m in generated_molecules if m is not None]
valid_smiles = [Chem.MolToSmiles(m, canonical=True, isomericSmiles=True) for m in valid_mols]

subset_mols = valid_mols[:200]
subset_smiles = valid_smiles[:200]
legends = [f"{i}\n{smi}" for i, smi in enumerate(subset_smiles)]

img = Draw.MolsToGridImage(
    subset_mols,
    molsPerRow=10,
    subImgSize=(200, 200),
    legends=legends,
    maxMols=1000,
    useSVG=False,
    returnPNG=False,
)

os.makedirs("images", exist_ok=True)
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

out_path = f"images/{TASK}_generated_molecules_{timestamp}.png"

img.save(out_path)
print(f"Saved image at -> {out_path}")
display(img)